## Лабораторная работа №2

1) Создание одного нейрона для задач AND и OR.
2) Создание нейронной сети из 10 нейронов для датасета MNIST. Метрика MSE. Необходимо добиться точности больше 50% для каждого класса. Sigmoid

### 1 часть

In [2]:
import numpy as np

X = np.array([[0,0], [0,1], [1,0], [1,1]], dtype=float)
y_and = np.array([0,0,0,1], dtype=float)
y_or = np.array([0,1,1,1], dtype=float)

def train(X, y, lr=0.1, epochs=20):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(epochs):
        for xi, yi in zip(X, y):
            pred = 1 if xi @ w + b >= 0 else 0
            w += lr * (yi - pred) * xi
            b += lr * (yi - pred)
    return w, b

def test(name, X, y, w, b):
    print(name, "w =", w, "b =", round(b,2))
    for xi, yi in zip(X, y):
        print(" ", xi, "->", 1 if xi @ w + b >= 0 else 0, "| ожидалось", yi)

test("AND", X, y_and, *train(X, y_and))
test("OR", X, y_or, *train(X, y_or))

AND w = [0.2 0.1] b = -0.2
  [0. 0.] -> 0 | ожидалось 0.0
  [0. 1.] -> 0 | ожидалось 0.0
  [1. 0.] -> 0 | ожидалось 0.0
  [1. 1.] -> 1 | ожидалось 1.0
OR w = [0.1 0.1] b = -0.1
  [0. 0.] -> 0 | ожидалось 0.0
  [0. 1.] -> 1 | ожидалось 1.0
  [1. 0.] -> 1 | ожидалось 1.0
  [1. 1.] -> 1 | ожидалось 1.0


### 2 часть

In [4]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

X, y = fetch_openml("mnist_784", version=1, return_X_y=True,
                    as_frame=False, parser="auto")
X = X.astype(np.float32) / 255.0
y = y.astype(int)
T = np.eye(10)[y] # OHE таргета
X_train, X_test, T_train, T_test, y_train, y_test = train_test_split(X, T, y, test_size=10000, random_state=0, stratify=y)

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

rng = np.random.default_rng(0)
W = rng.normal(0, 0.01, (784, 10))
b = np.zeros(10)

lr, epochs, bs = 1.0, 30, 64
n = len(X_train)

for ep in range(epochs):
    idx = rng.permutation(n)
    for s in range(0, n, bs):
        i = idx[s:s+bs]
        x, t = X_train[i], T_train[i]

        a = sigmoid(x @ W + b)
        d = (a - t) * a * (1 - a) / len(i)

        W -= lr * (x.T @ d)
        b -= lr * d.sum(0)

    a = sigmoid(X_train @ W + b)
    mse = np.mean((a - T_train) ** 2)
    acc = (a.argmax(1) == y_train).mean()
    # print(f"Эпоха {ep + 1:2d} MSE {mse:.5f} acc {acc:.4f}")

a = sigmoid(X_test @ W + b)
pred = a.argmax(1)

print(f"\nTest MSE {np.mean((a-T_test) ** 2):.5f}")
print(f"Test accuracy {(pred == y_test).mean():.4f}\n")

for c in range(10):
    m = y_test == c
    print(f"Класс {c}: {(pred[m] == c).mean() * 100:5.1f}% ({m.sum()} шт.)")


Test MSE 0.01532
Test accuracy 0.9204

Класс 0:  97.4% (986 шт.)
Класс 1:  97.3% (1125 шт.)
Класс 2:  90.8% (999 шт.)
Класс 3:  88.9% (1020 шт.)
Класс 4:  94.7% (975 шт.)
Класс 5:  85.9% (902 шт.)
Класс 6:  96.1% (982 шт.)
Класс 7:  93.8% (1042 шт.)
Класс 8:  88.9% (975 шт.)
Класс 9:  85.4% (994 шт.)
